## Cell 1 — Imports and constants

In [1]:
"""Phase 2.2 — Classical ML baselines on the same scikit-learn splits.

We train TWO linear classifiers on TF-IDF features and compare them
side-by-side with the DistilBERT result from Phase 2.1.

Goal: a calibration point. If DistilBERT only beats classical by a hair,
the fine-tune cost wasn't justified. If it wins by 5+ points of macro-F1,
we earned the GPU time.
"""
import json
import os
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import wandb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.svm import LinearSVC

os.environ["WANDB__SERVICE_WAIT"] = "300"

DATA_DIR = Path("..") / "data" / "issues" / "splits"
LABELS = ["bug", "feature", "docs", "question"]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

SEED = 42
np.random.seed(SEED)

## Cell 2 — Load splits

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open() as f:
        return [json.loads(line) for line in f]


def issue_to_text(row: dict) -> str:
    """Title + body, same as the DistilBERT side."""
    title = (row.get("title") or "").strip()
    body = (row.get("body") or "").strip()
    return f"{title}\n\n{body}" if body else title


train_rows = load_jsonl(DATA_DIR / "train.jsonl")
val_rows = load_jsonl(DATA_DIR / "val.jsonl")
test_rows = load_jsonl(DATA_DIR / "test.jsonl")

X_train_text = [issue_to_text(r) for r in train_rows]
y_train = np.array([LABEL_TO_ID[r["label"]] for r in train_rows])

X_val_text = [issue_to_text(r) for r in val_rows]
y_val = np.array([LABEL_TO_ID[r["label"]] for r in val_rows])

X_test_text = [issue_to_text(r) for r in test_rows]
y_test = np.array([LABEL_TO_ID[r["label"]] for r in test_rows])

print(f"train: {len(X_train_text)}")
print(f"val:   {len(X_val_text)}")
print(f"test:  {len(X_test_text)}")
print(f"\ntrain distribution: {dict(Counter(r['label'] for r in train_rows))}")

train: 2690
val:   576
test:  578

train distribution: {'feature': 798, 'docs': 673, 'bug': 1023, 'question': 196}


## Cell 3 — TF-IDF vectorization

In [3]:
# TF-IDF: word n-grams up to 2. The 2-gram captures things like
# "raises error", "fit fails", "documentation example" — useful for our classes.
# min_df=2 drops words that appear in fewer than 2 documents (typos, noise).
# max_df=0.95 drops words appearing in >95% of documents (stopword-ish noise).
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,  # log-scale term frequencies — standard for text classification
    lowercase=True,
    strip_accents="unicode",
)

# Fit on train ONLY. val and test get .transform() but never .fit().
X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

print(f"vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"train feature matrix: {X_train.shape}")
print(f"val feature matrix:   {X_val.shape}")
print(f"test feature matrix:  {X_test.shape}")

vocabulary size: 42225
train feature matrix: (2690, 42225)
val feature matrix:   (576, 42225)
test feature matrix:  (578, 42225)


## Cell 4 — Initialize W&B

In [4]:
wandb.init(
    project="maintainer-copilot",
    name=f"classical-baseline-{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}",
    job_type="baseline",
    config={
        "approach": "tfidf+linear",
        "vectorizer": {
            "ngram_range": [1, 2],
            "min_df": 2,
            "max_df": 0.95,
            "sublinear_tf": True,
            "vocab_size": len(vectorizer.vocabulary_),
        },
        "labels": LABELS,
        "train_size": len(X_train_text),
        "val_size": len(X_val_text),
        "test_size": len(X_test_text),
        "seed": SEED,
    },
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/bmislol/.netrc.
wandb: Currently logged in as: bmislol (bmislol-se-factory) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Cell 5 — Train LogisticRegression

In [6]:
# class_weight="balanced" mirrors what DistilBERT did with explicit class weights.
# C is the inverse regularization strength; we use the default 1.0 but log it.
logreg = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    solver="lbfgs",
    random_state=SEED,
    max_iter=2000,
)
logreg.fit(X_train, y_train)

logreg_val_preds = logreg.predict(X_val)
logreg_val_acc = accuracy_score(y_val, logreg_val_preds)
logreg_val_f1 = f1_score(y_val, logreg_val_preds, average="macro")
print(f"LogReg  val acc: {logreg_val_acc:.4f}  macro-F1: {logreg_val_f1:.4f}")

wandb.log({"logreg/val/accuracy": logreg_val_acc, "logreg/val/macro_f1": logreg_val_f1})

LogReg  val acc: 0.7396  macro-F1: 0.6473


## Cell 6 — Train LinearSVC

In [7]:
# LinearSVC has no predict_proba directly. We use it because it often
# outperforms LogReg on TF-IDF text classification by a small margin.
svc = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=SEED,
    max_iter=2000,
    dual="auto",
)
svc.fit(X_train, y_train)

svc_val_preds = svc.predict(X_val)
svc_val_acc = accuracy_score(y_val, svc_val_preds)
svc_val_f1 = f1_score(y_val, svc_val_preds, average="macro")
print(f"LinSVC  val acc: {svc_val_acc:.4f}  macro-F1: {svc_val_f1:.4f}")

wandb.log({"svc/val/accuracy": svc_val_acc, "svc/val/macro_f1": svc_val_f1})

LinSVC  val acc: 0.7396  macro-F1: 0.6261


## Cell 7 — Pick winner on val, then evaluate ONCE on test

In [8]:
# Pick the better model on val. Test set stays untouched until now.
if logreg_val_f1 >= svc_val_f1:
    winner = logreg
    winner_name = "LogisticRegression"
else:
    winner = svc
    winner_name = "LinearSVC"

print(f"\nWinner on val: {winner_name}\n")

# Final test evaluation — ONCE.
test_preds = winner.predict(X_test)
test_accuracy = accuracy_score(y_test, test_preds)
test_macro_f1 = f1_score(y_test, test_preds, average="macro")

per_class_report = classification_report(
    y_test,
    test_preds,
    target_names=LABELS,
    output_dict=True,
    zero_division=0,
)
per_class_f1 = {label: per_class_report[label]["f1-score"] for label in LABELS}
cm = confusion_matrix(y_test, test_preds, labels=list(range(len(LABELS))))

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test macro-F1: {test_macro_f1:.4f}")
print("\nPer-class F1:")
for label, f1 in per_class_f1.items():
    print(f"  {label:10s}: {f1:.4f}")
print(f"\nConfusion matrix (rows=true, cols=pred), order: {LABELS}")
print(cm)

wandb.log(
    {
        "test/accuracy": test_accuracy,
        "test/macro_f1": test_macro_f1,
        "test/winner": winner_name,
        **{f"test/f1_{label}": per_class_f1[label] for label in LABELS},
    }
)


Winner on val: LogisticRegression

Test accuracy: 0.8201
Test macro-F1: 0.6977

Per-class F1:
  bug       : 0.8961
  feature   : 0.7826
  docs      : 0.8562
  question  : 0.2558

Confusion matrix (rows=true, cols=pred), order: ['bug', 'feature', 'docs', 'question']
[[263   0   6   5]
 [ 12  72   1   4]
 [ 16   5 128   0]
 [ 22  18  15  11]]


## Cell 8 — Side-by-side comparison with DistilBERT

In [9]:
# Load Phase 2.1's model card for the side-by-side comparison.
distilbert_card = json.loads(
    (Path("..") / "data" / "classifier_artifacts" / "model_card.json").read_text()
)

print(f"\n{'=' * 60}")
print("CLASSIFIER COMPARISON (test set)")
print(f"{'=' * 60}")
print(f"{'Metric':<22} {'DistilBERT':>13} {winner_name + ' (classical)':>22}")
print(f"{'-' * 60}")
print(f"{'Test accuracy':<22} {distilbert_card['test_accuracy']:>13.4f} {test_accuracy:>22.4f}")
print(f"{'Test macro-F1':<22} {distilbert_card['test_macro_f1']:>13.4f} {test_macro_f1:>22.4f}")
for label in LABELS:
    distil_f1 = distilbert_card["per_class_f1"][label]
    print(f"{'F1 ' + label:<22} {distil_f1:>13.4f} {per_class_f1[label]:>22.4f}")

# Compute the gap.
delta = distilbert_card["test_macro_f1"] - test_macro_f1
print(f"\nDistilBERT advantage: {delta:+.4f} macro-F1 ({100 * delta / test_macro_f1:+.1f}%)")


CLASSIFIER COMPARISON (test set)
Metric                    DistilBERT LogisticRegression (classical)
------------------------------------------------------------
Test accuracy                 0.8478                 0.8201
Test macro-F1                 0.7462                 0.6977
F1 bug                        0.9255                 0.8961
F1 feature                    0.8148                 0.7826
F1 docs                       0.8845                 0.8562
F1 question                   0.3600                 0.2558

DistilBERT advantage: +0.0485 macro-F1 (+7.0%)


## Cell 9 — Save artifacts and finish

In [ ]:
import pickle

ARTIFACT_DIR = Path("..") / "data" / "classical_baseline_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Persist the trained vectorizer + winner model — small (~5 MB total).
# Not pushed to MinIO. Lives on disk for reproducibility and Phase 2.3
# can load it for the three-way comparison.
with (ARTIFACT_DIR / "vectorizer.pkl").open("wb") as f:
    pickle.dump(vectorizer, f)
with (ARTIFACT_DIR / "classifier.pkl").open("wb") as f:
    pickle.dump(winner, f)

# Comparison report — readable summary for DECISIONS.md and the model_card analog.
report = {
    "approach": "tfidf+linear",
    "winner": winner_name,
    "vectorizer": {
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True,
        "vocab_size": len(vectorizer.vocabulary_),
    },
    "classifier": {
        "name": winner_name,
        "C": 1.0,
        "class_weight": "balanced",
        "max_iter": 2000,
        "seed": SEED,
    },
    "val_comparison": {
        "logreg_macro_f1": float(logreg_val_f1),
        "svc_macro_f1": float(svc_val_f1),
    },
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "per_class_f1": per_class_f1,
    "confusion_matrix": cm.tolist(),
    "compared_to_distilbert": {
        "distilbert_test_accuracy": distilbert_card["test_accuracy"],
        "distilbert_test_macro_f1": distilbert_card["test_macro_f1"],
        "distilbert_per_class_f1": distilbert_card["per_class_f1"],
        "delta_macro_f1": float(delta),
    },
    "trained_at": datetime.now(UTC).isoformat(),
}

(ARTIFACT_DIR / "comparison_report.json").write_text(json.dumps(report, indent=2))
print(f"saved {ARTIFACT_DIR / 'comparison_report.json'}")

wandb.finish()

saved ../data/classical_baseline_artifacts/comparison_report.json


logreg/val/accuracy,▁
logreg/val/macro_f1,▁
svc/val/accuracy,▁
svc/val/macro_f1,▁
test/accuracy,▁
test/f1_bug,▁
test/f1_docs,▁
test/f1_feature,▁
test/f1_question,▁
test/macro_f1,▁
logreg/val/accuracy,0.73958
